# MST construction scaling: exact Prim vs FAMST

This notebook compares **MST graph construction only** for exact dense Prim and the sparse approximate FAMST backend as the dataset grows. Both methods receive the same synthetic data for each size. The embedding optimization is intentionally excluded so this measures the MST backends directly.

FAMST is approximate. Alongside wall-clock speedup, the notebook reports relative total tree-weight error against Prim wherever the exact run fits the configured memory budget. FAMST needs the optional dependency: `python -m pip install -e ".[famst]"`. Run from the repository root.

The timings are machine- and environment-dependent. The first small FAMST run warms its JIT-compiled neighbor-search code and is excluded from the measurements.

In [1]:
from time import perf_counter
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from imste.mst import build_mst_edges

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")

ModuleNotFoundError: No module named 'imste'

## Benchmark settings

Increase `SAMPLE_SIZES` to explore larger inputs. Exact Prim is skipped when its estimated distance-matrix and packed-edge storage exceeds `EXACT_MEMORY_BUDGET_GB`; this is a guard estimate, not a system memory monitor. Keep `REPEATS` small for larger sizes because exact Prim has quadratic cost.

In [ ]:
SAMPLE_SIZES = [128, 256, 512, 1_024, 2_048, 4_096]
N_FEATURES = 32
N_MSTS = 1
ANN_NEIGHBORS = 15
INTER_COMPONENT_EDGES = 5
MAX_ANN_NEIGHBORS = 60
REPEATS = 2
DATA_SEED = 42
BENCHMARK_SEED = 123
EXACT_MEMORY_BUDGET_GB = 1.0

def estimated_prim_graph_bytes(n_samples):
    # float64 pairwise distances plus the packed edge-availability matrix.
    return n_samples**2 * (8 + 1 / 8)

pd.DataFrame({
    'n_samples': SAMPLE_SIZES,
    'estimated_exact_graph_memory_mb': [
        estimated_prim_graph_bytes(n) / 1024**2 for n in SAMPLE_SIZES
    ],
})

## Warm up the backends

FAMST uses PyNNDescent, which compiles some routines on first use. This small warm-up is not included in the benchmark.

In [ ]:
warmup_X = np.random.default_rng(DATA_SEED).normal(size=(64, N_FEATURES))

_ = build_mst_edges(
    warmup_X, n_msts=N_MSTS, random_state=BENCHMARK_SEED,
    method='prim', neighbors=ANN_NEIGHBORS,
    inter_component_edges=INTER_COMPONENT_EDGES,
    max_neighbors=MAX_ANN_NEIGHBORS,
)
_ = build_mst_edges(
    warmup_X, n_msts=N_MSTS, random_state=BENCHMARK_SEED,
    method='famst', neighbors=ANN_NEIGHBORS,
    inter_component_edges=INTER_COMPONENT_EDGES,
    max_neighbors=MAX_ANN_NEIGHBORS,
)
print('Warm-up complete.')

## Run the benchmark

Each repeat uses the same dataset for both methods. The reported duration is the median across repeats. FAMST's random seed is fixed for repeatability.

In [ ]:
records = []

for n_samples in SAMPLE_SIZES:
    X = np.random.default_rng(DATA_SEED + n_samples).normal(
        size=(n_samples, N_FEATURES)
    )
    exact_memory_gb = estimated_prim_graph_bytes(n_samples) / 1024**3
    run_exact = exact_memory_gb <= EXACT_MEMORY_BUDGET_GB

    methods = ['famst', 'prim'] if run_exact else ['famst']
    for method in methods:
        durations = []
        tree_weight = None
        n_edges = None
        for repeat in range(REPEATS):
            started = perf_counter()
            edges, weights = build_mst_edges(
                X, n_msts=N_MSTS,
                random_state=BENCHMARK_SEED + repeat,
                method=method,
                neighbors=ANN_NEIGHBORS,
                inter_component_edges=INTER_COMPONENT_EDGES,
                max_neighbors=MAX_ANN_NEIGHBORS,
            )
            durations.append(perf_counter() - started)
            tree_weight = float(weights.sum())
            n_edges = len(edges)

        records.append({
            'n_samples': n_samples,
            'method': method,
            'median_seconds': float(np.median(durations)),
            'repeat_seconds': durations,
            'tree_weight': tree_weight,
            'n_edges': n_edges,
            'exact_memory_estimate_gb': exact_memory_gb,
            'exact_skipped': not run_exact,
        })

results = pd.DataFrame(records)
results

## Compare runtime and approximate-tree weight

Speedup is `Prim time / FAMST time`; values above 1 mean FAMST was faster. Relative weight error compares total edge lengths for the same dataset. It is a useful approximation check, but it does not measure whether the selected edges are identical.

In [ ]:
wide = results.pivot(index='n_samples', columns='method', values='median_seconds')
summary = pd.DataFrame(index=wide.index)
summary['prim_seconds'] = wide.get('prim')
summary['famst_seconds'] = wide.get('famst')
summary['speedup_prim_over_famst'] = (
    summary['prim_seconds'] / summary['famst_seconds']
)
weights = results.pivot(index='n_samples', columns='method', values='tree_weight')
if {'prim', 'famst'}.issubset(weights.columns):
    summary['famst_relative_weight_error_pct'] = (
        100 * (weights['famst'] - weights['prim']) / weights['prim']
    )
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for method, group in results.groupby('method'):
    axes[0].plot(
        group['n_samples'], group['median_seconds'],
        marker='o', label=method.upper(),
    )
axes[0].set(xscale='log', yscale='log', xlabel='Number of rows',
            ylabel='MST construction time (seconds)',
            title='Graph construction time')
axes[0].grid(True, which='both', alpha=0.25)
axes[0].legend()

available_speedups = summary['speedup_prim_over_famst'].dropna()
axes[1].plot(available_speedups.index, available_speedups,
             marker='o', color='darkgreen')
axes[1].axhline(1, color='black', linewidth=1, linestyle='--')
axes[1].set(xscale='log', xlabel='Number of rows',
            ylabel='Speedup (Prim / FAMST)', title='FAMST speedup')
axes[1].grid(True, which='both', alpha=0.25)
plt.tight_layout()
plt.show()

## Interpretation

- At small sizes, ANN indexing and component handling can outweigh the savings from avoiding a dense distance matrix.
- At larger sizes, FAMST should benefit from its sparse neighbor graph; the crossover point depends on data, dimensions, hardware, and ANN settings.
- If Prim is skipped, no speedup or exact weight-error value is reported for that size.
- The exact backend is the reference for this synthetic dataset. FAMST can have low total-weight error while still choosing different tree edges.